In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client
from tqdm import tqdm
import time
import sys

# --- CONFIGURATION ---
INPUT_FILE_NAME = 'S1_D.xlsx'  
OUTPUT_FILE_NAME = 'output_drugs.xlsx'
CHEMBL_ID_COLUMN = 'ChEMBL_ID'  # Ensure this column name matches your Excel file

# Mapping Keywords
INHIBITION_KEYWORDS = [
    'inhibitor', 'antagonist', 'blocker', 'suppressor', 'inactivator', 
    'inverse agonist', 'negative modulator', 'reducer', 'disruptor'
]

ACTIVATION_KEYWORDS = [
    'agonist', 'activator', 'stimulator', 'up-regulator', 'positive modulator', 
    'inducer', 'enhancer', 'stabiliser'
]

def determine_mechanism_of_action(chembl_id):
    """
    Queries the ChEMBL database for a given molecule ID to ascertain its mechanism of action.
    Returns a tuple: (descriptive string, numerical value for ASP)
    """
    if pd.isna(chembl_id) or str(chembl_id).strip() == '':
        return "Missing ID", None
    
    chembl_id = str(chembl_id).strip()
    
    try:
        mechanisms = new_client.mechanism.filter(
            molecule_chembl_id=chembl_id
        ).only(['action_type', 'mechanism_of_action'])
        
        if not mechanisms:
            return "Mechanism Not Found", None

        actions_found = []
        for mech in mechanisms:
            if mech.get('action_type'):
                actions_found.append(mech['action_type'].lower())
        
        if not actions_found:
            return "Mechanism Not Found", None

        is_inhibitor = any(k in action for action in actions_found for k in INHIBITION_KEYWORDS)
        is_activator = any(k in action for action in actions_found for k in ACTIVATION_KEYWORDS)

        if is_inhibitor and not is_activator:
            return "Inhibition", 0
        elif is_activator and not is_inhibitor:
            return "Activation", 1
        elif is_inhibitor and is_activator:
            return "Mixed/Complex", None 
        else:
            return "Other/Binding", None

    except Exception as e:
        return "API Error", None

def main():
    print("--- Initialization of Data Enrichment Process ---")
    
    try:
        df = pd.read_excel(INPUT_FILE_NAME, engine='openpyxl')
        print(f"File loaded successfully: {len(df)} rows detected.")
    except Exception as e:
        print(f"ERROR: Unable to open file: {e}")
        return

    target_col = CHEMBL_ID_COLUMN
    if CHEMBL_ID_COLUMN not in df.columns:
        print(f"Warning: Column '{CHEMBL_ID_COLUMN}' not found. Defaulting to column index 2.")
        target_col = df.columns[2]

    compound_cache = {}
    action_results = []
    boolean_results = []

    print("Initiating ChEMBL API queries...")
    
    try:
        for index, row in tqdm(df.iterrows(), total=df.shape[0]):
            chembl_id = row[target_col]
            
            if chembl_id in compound_cache:
                action, value = compound_cache[chembl_id]
            else:
                action, value = determine_mechanism_of_action(chembl_id)
                compound_cache[chembl_id] = (action, value)
            
            action_results.append(action)
            boolean_results.append(value)

        # Rename first three columns to English
        df.rename(columns={
            df.columns[0]: "Gene",
            df.columns[1]: "Drug",
            df.columns[2]: "ChEMBL_ID"
        }, inplace=True)

        # Add translated columns
        df["Action"] = action_results
        df["Boolean_Value"] = boolean_results
        
        print("Data processing completed successfully.")

    except KeyboardInterrupt:
        print("\nManual interruption detected.")
        df = df.iloc[:len(action_results)].copy()
        df["Action"] = action_results
        df["Boolean_Value"] = boolean_results

    except Exception as e:
        print(f"\nUnexpected Error: {e}")
        df = df.iloc[:len(action_results)].copy()
        df["Action"] = action_results
        df["Boolean_Value"] = boolean_results

    finally:
        print(f"Saving output file to: {OUTPUT_FILE_NAME}...")
        df.to_excel(OUTPUT_FILE_NAME, index=False, engine='openpyxl')
        print("File serialization completed.")

if __name__ == "__main__":
    main()
